# research-paper-lora v2: retrain on the updated 53-example dataset

This notebook documents the second training run, after the dataset was updated to replace a rice-area-mapping paper (Mohite et al.) with two additional papers (apple orchard canopy characterization and FAIMS off-flavor detection), bringing the dataset from 42 to 53 examples. All outputs below are real and unedited.


In [1]:
!pip install -q transformers peft trl bitsandbytes datasets accelerate


## Confirm GPU runtime

In [3]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory / 1e9, "GB VRAM")


## Upload the updated (53-example) dataset

Restarted the runtime before this step to clear a stale `qa_seed.jsonl` left over from the previous run, which had silently caused the first v2 attempt to train on the old 42-example dataset.

In [5]:
from google.colab import files
uploaded = files.upload()  # select the updated qa_seed.jsonl (53 pairs)


Saving qa_seed.jsonl to qa_seed.jsonl


In [6]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="qa_seed.jsonl", split="train")
print(len(dataset))
print(dataset[0])


53
{'instruction': 'What sensing technology was evaluated for detecting Little Cherry Disease and X-disease in sweet cherry?', 'input': '', 'output': "Field asymmetric ion mobility spectrometry (FAIMS) was evaluated for rapid, non-invasive detection of Little Cherry Disease (LCD) and X-disease in three sweet cherry cultivars ('Benton', 'Cristalina', and 'Tieton') at the post-harvest stage, using stem cuttings with leaves collected from commercial orchards and greenhouse trees."}


## Load base model in 4-bit (Qwen2.5-1.5B-Instruct)

In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)


In [9]:
def format_example(example):
    messages = [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

formatted_dataset = dataset.map(format_example)


## Eval harness + baseline (before fine-tuning on the updated dataset)

Eval questions updated: the rice-mapping question was swapped for one about the apple orchard canopy volume study.

In [11]:
def ask(model, tokenizer, question, max_new_tokens=150):
    messages = [{"role": "user", "content": question}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    input_len = inputs["input_ids"].shape[1]
    return tokenizer.decode(out[0][input_len:], skip_special_tokens=True)

eval_questions = [
    "What accuracy did Extra Trees achieve for Atlantic salmon maturity classification from hyperspectral data?",
    "How much did mAP50 vary across the 12 YOLO model variants in the aquaculture edge benchmark?",
    "Which 3D surface reconstruction method correlated best with manual canopy volume measurements in the apple orchard study, and what was the correlation coefficient?",
]

baseline_answers = {q: ask(model, tokenizer, q) for q in eval_questions}
for q, a in baseline_answers.items():
    print("Q:", q, "\nA:", a, "\n")


Q: What accuracy did Extra Trees achieve for Atlantic salmon maturity classification from hyperspectral data? 
A: I'm unable to provide specific numerical results or details about the performance of machine learning models in this context without access to proprietary information. However, I can tell you that ExtraTrees is an ensemble method used for regression and classification tasks, which typically involves training multiple decision trees on different subsets of the dataset (bootstrap samples) and then combining their predictions.

For the task of classifying Atlantic salmon maturity based on hyperspectral data, the accuracy would depend on several factors including the quality and quantity of the data, the complexity of the model, and the tuning parameters. The ExtraTrees algorithm itself does not have a fixed accuracy rate; its performance varies depending on how well it generalizes from the training data to unseen data.

If you need more precise information, consulting 

Q: How

**Result: base model correctly declines on all three questions**, including the new apple orchard question — consistent, honest baseline behavior.

## LoRA fine-tune, 10 epochs, on the correct 53-example dataset

In [14]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [15]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./qwen-research-lora-v2",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=3e-4,
    logging_steps=5,
    save_strategy="epoch",
    bf16=True,
    report_to="none",
    dataset_text_field="text",
    max_length=512,
)

trainer = SFTTrainer(model=model, args=training_args, train_dataset=formatted_dataset)
trainer.train()


TrainOutput(global_step=70, training_loss=0.7932063124009541, metrics={'train_runtime': 414.7812, 'train_samples_per_second': 1.278, 'train_steps_per_second': 0.169, 'total_flos': 559373911080960.0, 'train_loss': 0.7932063124009541, 'epoch': 10.0})


In [16]:
model.eval()
model.gradient_checkpointing_disable()
model.config.use_cache = True

after_answers = {q: ask(model, tokenizer, q) for q in eval_questions}
for q, a in after_answers.items():
    print("Q:", q, "\nA:", a, "\n")


Q: What accuracy did Extra Trees achieve for Atlantic salmon maturity classification from hyperspectral data? 
A: The Extra Trees algorithm achieved 82.4% accuracy on the full-spectrum dataset (Dataset 1), and when trained on the same features but applied to the FAIMS compressed dataset (Dataset 3) it achieved 79.7% accuracy, demonstrating that feature selection via extra trees strength can yield comparable performance to manual feature engineering. 

Q: How much did mAP50 vary across the 12 YOLO model variants in the aquaculture edge benchmark? 
A: Model variants were trained on different image sizes (480x480 and 640x640) and differed by their top-layer architecture (Darknet-50, Darknet-101, Resnext-14, and Resnext-16). Within each architecture, models were sorted by accuracy, with the most accurate 15% of images assigned to mAP50 only, leaving the rest to mAP50++. 

Q: Which 3D surface reconstruction method correlated best with manual canopy volume measurements in the apple orchard s

**Result: mixed.** Q3 (apple orchard) is essentially correct — convex hull r=0.64 and alpha-shape r=0.62 both match the paper exactly; only the voxel-grid figure is off (0.35 vs. actual 0.25). Q1 (salmon) partially confabulates: 82.4% is close to the real 81.8%, but the "FAIMS compressed dataset / Dataset 3 / 79.7%" detail doesn't exist in the source paper or training data. Q2 (YOLO26) is fully fabricated — none of "Darknet-50/101," "Resnext-14/16," different image sizes, or "mAP50++" appear anywhere in the paper or training data; the model invented plausible-sounding ML jargon from scratch.

In [18]:
train_q = dataset[10]["instruction"]
train_a = dataset[10]["output"]
print("Ground truth:", train_a)
print("\nModel's answer:", ask(model, tokenizer, train_q))


Ground truth: Extra Trees performed best overall, achieving 81.8% accuracy on the full-spectrum dataset (Dataset 1), and maintained strong performance (around 78%) on filtered and feature-selected dataset variants.

Model's answer: Extra Trees performed best overall, achieving 81.9% accuracy on the full-spectrum dataset (Dataset 1), and maintained strong performance (around 75%) on filtered and feature-selected dataset variants.


## Finding (v2)

On a question copied near-verbatim from training data, the model reproduces it almost exactly (81.9% vs. 81.8%; "around 75%" vs. "around 78%" — both close, minor drift). This confirms training worked on the real 53-example dataset.

On the three held-out questions (phrased differently from training examples), results varied by how much repetition that specific fact had in the dataset:

- **Apple orchard (Q3):** nearly perfect. This paper's key numbers (r=0.64, r=0.62) appeared clearly in its dedicated Q&A pairs and were reproduced almost exactly.
- **Salmon maturity (Q1):** partially correct accuracy figure, but an invented secondary detail not present anywhere in the source material.
- **YOLO26 (Q2):** complete fabrication — invented architecture names and metrics with no basis in the training data at all.

**Takeaway:** the same underfitting-to-confabulation pattern from the first dataset version replicated on this larger, updated dataset, but with an added nuance: how well a specific fact generalizes under paraphrase seems to depend on how much supporting detail existed for that paper in the training set, not just total dataset size. This is consistent with, and adds more evidence to, the standard fix already identified: more examples per fact, phrased multiple ways.